# 📘 03 — Dhara: Ingest Rulebook PDFs (Bronze → Silver chunks)

Parses both PDFs into structured chunks **with page and section metadata preserved** — this is what makes Vani's citations truthful.

- `Railway General Rules 1976` → operational rules
- `Railways Act 1989` → passenger rights (Sections 54, 55, 123, 124, 124A, 138, etc.)

Each chunk is ~500 chars with 50-char overlap, carrying page number, source, and detected section (Act sections are numbered like "124A" or "55").

In [0]:
%pip install -q pypdf
%restart_python

In [0]:
from pypdf import PdfReader
import re

VOLUME = "/Volumes/setu_rail/bronze/raw_files"

PDF_SPECS = [
    {"name": "rules_1976", "filename": "1976_general_rules_railways_pdf.pdf",
     "title": "General Rules for Indian Railways, 1976"},
    {"name": "act_1989",   "filename": "the_railways_act__1989.pdf",
     "title": "The Railways Act, 1989"},
]

# Regex to catch section headers like "124A." or "55." at the start of a line
SECTION_RE = re.compile(r"^\s*(\d{1,4}[A-Z]?)[\.\s]", re.MULTILINE)

def extract_chunks(path, source_name, source_title, chunk_size=500, overlap=50):
    """Yield dicts with {source, source_title, page, section, chunk_id, text}."""
    reader = PdfReader(path)
    for p_idx, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if len(text) < 30:
            continue

        # Detect any section numbers appearing on this page
        sections_on_page = SECTION_RE.findall(text)
        primary_section = sections_on_page[0] if sections_on_page else None

        # Sliding-window chunks
        step = chunk_size - overlap
        for offset in range(0, len(text), step):
            chunk_text = text[offset:offset + chunk_size]
            if len(chunk_text) < 80:
                continue
            yield {
                "source":        source_name,
                "source_title":  source_title,
                "page":          p_idx,
                "section":       primary_section,
                "chunk_id":      f"{source_name}_p{p_idx}_o{offset}",
                "text":          chunk_text,
            }

all_rows = []
for spec in PDF_SPECS:
    path = f"{VOLUME}/{spec['filename']}"
    print(f"📄 Parsing {spec['filename']} ...")
    rows = list(extract_chunks(path, spec["name"], spec["title"]))
    print(f"   → {len(rows)} chunks")
    all_rows.extend(rows)

print(f"\n📊 Total chunks across both PDFs: {len(all_rows):,}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("source",       StringType(), False),
    StructField("source_title", StringType(), False),
    StructField("page",         IntegerType(), False),
    StructField("section",      StringType(), True),
    StructField("chunk_id",     StringType(), False),
    StructField("text",         StringType(), False),
])

rules_df = spark.createDataFrame(all_rows, schema)

# Add an autoincrementing id for Vector Search primary key
from pyspark.sql.functions import monotonically_increasing_id
rules_df = rules_df.withColumn("id", monotonically_increasing_id())

(rules_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")   # Required for Vector Search sync
    .saveAsTable("setu_rail.bronze.rules_raw"))

# Also create the silver.rules_chunks view straightaway — no further cleaning needed
spark.sql("""
    CREATE OR REPLACE TABLE setu_rail.silver.rules_chunks
    USING DELTA
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
    AS SELECT id, source, source_title, page, section, chunk_id, text
    FROM setu_rail.bronze.rules_raw
""")

print("✅ bronze.rules_raw + silver.rules_chunks written (with CDF enabled for Vector Search)")

In [0]:
%sql
-- Sanity: how many chunks per source + a few passenger-rights sections visible
SELECT source, COUNT(*) AS chunk_count FROM setu_rail.silver.rules_chunks GROUP BY source;

In [0]:
%sql
-- Peek at passenger-rights-relevant sections (54=ticket, 124A=compensation for untoward incidents, 138=excess charge)
SELECT page, section, SUBSTRING(text, 1, 200) AS preview
FROM   setu_rail.silver.rules_chunks
WHERE  source = 'act_1989' AND section IN ('54', '55', '123', '124', '124A', '138')
ORDER BY page
LIMIT 10;

✅ **Next:** `04_build_silver_enriched.ipynb`